# Follow the Evolution of one codespace recovery after 1 step

In [150]:
using Pkg
Pkg.activate("../DecoKiller/")
Pkg.precompile();

  Activating project at `~/UNIPA/COLLISION_MODELS/shooting-decoherences/DecoKiller`


In [151]:
using Revise, StatsBase, Latexify
using DecoKiller

includet("../DecoKiller/src/UnitaryDilation/UnitaryDilation.jl")
using .UnitaryDilation

includet("../DecoKiller/src/PetzMaps.jl")
using .PetzMaps

includet("../DecoKiller/src/utils.jl")

using JLD2
using TOML


In [152]:

includet("../DecoKiller/scripts/main_iterator.jl")
experiments_dir = "../experiments/test_experiments"
main(["experiments_dir=$experiments_dir"])

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
 State 99 100%|█████████████████████████████████████████| Time: 0:00:00
    Fidelity: 0.6311442473221226
Adaptive Recovery 100%|█████████████████████████████████| Time: 0:00:07


└  Average fidelity (10 steps) = 0.5953540585310223

✓  All 8 experiment(s) complete.
   Summary → ../experiments/test_experiments/iterations_summary.csv


In [ ]:
folder=joinpath(pwd(), experiments_dir, "steps_n3_b08_n_timesteps10")
maps = load(joinpath(folder, "data", "superoperators.jld2"))["maps"];
d2=Int(sqrt(length(maps[1]["Cx"])))

resh(x) = reshape(x, d2, d2)

cfg,st,lgs=DecoKiller.load_configuration(joinpath(folder,"config.toml"))

maps1 = maps[1:10]

r1 = st.ρ0

Using specified real noise from config: bitflip



8×8 Matrix{ComplexF64}:
 0.162278+0.0im       0.0+0.0im  0.0+0.0im  …  0.0+0.0im  0.102683+0.354119im
      0.0+0.0im       0.0+0.0im  0.0+0.0im     0.0+0.0im       0.0+0.0im
      0.0+0.0im       0.0+0.0im  0.0+0.0im     0.0+0.0im       0.0+0.0im
      0.0+0.0im       0.0+0.0im  0.0+0.0im     0.0+0.0im       0.0+0.0im
      0.0+0.0im       0.0+0.0im  0.0+0.0im     0.0+0.0im       0.0+0.0im
      0.0+0.0im       0.0+0.0im  0.0+0.0im  …  0.0+0.0im       0.0+0.0im
      0.0+0.0im       0.0+0.0im  0.0+0.0im     0.0+0.0im       0.0+0.0im
 0.102683-0.354119im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.837722+0.0im

In [154]:
function evolve(ρ, M::Vector{ComplexF64})
  ρ_new = unvec(resh(M) * vec(ρ))
  return ρ_new / tr(ρ_new)
end

r1_ = evolve(r1, maps1[1]["Nx"])
rho1_ = evolve(r1, maps1[1]["N1"])
rho2_ = evolve(r1, maps1[1]["N2"])
r1_rec = evolve(r1_, maps[1]["P"])
rho1_rec = evolve(rho1_, maps[1]["P"])
rho2_rec = evolve(rho2_, maps[1]["P"])
println(DecoKiller.fidelity(st.ρ0, r1_))
println(DecoKiller.fidelity(st.ρ0, r1_rec))

0.22569931732776416
0.5183709739883372


In [155]:
r1 = st.ρ0
r1_ = evolve(r1, maps1[1]["Nx"])
r1_rec = evolve(r1_, maps[1]["P"])
r2 = unvec(resh(maps1[1]["P"])*resh(maps1[1]["Nx"])*vec(r1))
isapprox(r2/tr(r2), r1_rec; atol=1e-14)

true

In [156]:
r2 = copy(r1_rec)
rho1 = copy(rho1_rec)
rho2 = copy(rho2_rec)

# Step 2: Informative collision
η = cfg.ancilla_state
da = size(η, 1)
ds = 2^cfg.n_qubits

model = CollisionModel(cfg.collision_unitary, cfg.sigma, ds, da, ancilla_state=η)
real_kraus = cfg.real_noise.extended_kraus
option_kraus = [noise.extended_kraus for noise in st.noise_options]

# Get the composite state system+ancilla
rho_to_rec_ = apply_collision(model, r2; ancilla_state=η, trace=false)
rho1_ = apply_collision(model, rho1; ancilla_state=η, trace=false)
rho2_ = apply_collision(model, rho2; ancilla_state=η, trace=false)

# Apply noise only to the system
n_subsystems = cfg.correlated_noise ? 1 : cfg.n_qubits
rho_to_rec_ = apply_extended_channel(rho_to_rec_, real_kraus; dim_to_extend=da)
rho1_ = apply_extended_channel(rho1_, option_kraus[1]; dim_to_extend=da)
rho2_ = apply_extended_channel(rho2_, option_kraus[2]; dim_to_extend=da)

# Create the supermap corresponding to the collision followed by the noise
Xi = kraus_to_superop(compose_kraus(real_kraus, model.kraus_fwd))
Xi1 = kraus_to_superop(compose_kraus(option_kraus[1], model.kraus_fwd))
Xi2 = kraus_to_superop(compose_kraus(option_kraus[2], model.kraus_fwd))

rho_to_rec_from_superop = evolve(r2, Xi)
rho_to_rec_from_superop /= tr(rho_to_rec_from_superop)
isapprox(rho_to_rec_from_superop, ptrace_ancilla(rho_to_rec_, ds, da); atol=1e-14)

true

In [157]:
# Step 3: Measurement
q1 = st.noise_options[1].probability
q2 = st.noise_options[2].probability
w, Πs = DecoKiller.discrimin(rho_to_rec_, rho1_, rho2_, ds, da, q1, q2)
povm = sample(cfg.rng, [1, 2], Weights(w))

1

In [158]:
# Collapse the system and trace out the ancilla
weakness = cfg.povm_weakness
Π = weakness * I(size(Πs[povm], 1)) / 2 + (1 - weakness) * Πs[povm]
println("Measurement: ")
display(Π)
rho_to_rec = ptrace_ancilla(DecoKiller.collapse_state(rho_to_rec_, Π), ds, da)
rho1 = ptrace_ancilla(DecoKiller.collapse_state(rho1_, Π), ds, da)
rho2 = ptrace_ancilla(DecoKiller.collapse_state(rho2_, Π), ds, da)

Measurement: 


2×2 Matrix{ComplexF64}:
 0.5+0.0im  0.0+0.0im
 0.0+0.0im  0.5+0.0im

8×8 Matrix{ComplexF64}:
     0.133295+2.3392e-35im   …     0.0286594+0.0065608im
 -1.37806e-17-4.56027e-19im     -9.45083e-19+8.12155e-19im
 -4.55803e-20+3.90333e-20im     -5.06346e-18-3.78477e-20im
 -5.79742e-18+1.18939e-19im      2.51322e-18+1.43954e-19im
  3.27654e-18-1.03542e-19im     -5.30044e-18-9.3804e-20im
 -6.29564e-18+1.32051e-20im  …   6.37935e-19-9.12625e-20im
   -1.057e-18-9.54217e-19im     -1.00304e-17+4.66662e-19im
    0.0286594-0.0065608im           0.145809-2.41335e-35im

In [159]:
println("If this is true, then the collapse is the identity")
println(isapprox(DecoKiller.collapse_state(rho_to_rec_, Π), rho_to_rec_; atol=1e-12))
# Get the CPTP map corresponding to the collapse
Cx = DecoKiller.collapse_map(ptrace_ancilla(rho_to_rec_, ds, da), rho_to_rec; pin=cfg.pin)
C1 = DecoKiller.collapse_map(ptrace_ancilla(rho1_, ds, da), rho1; pin=cfg.pin)
C2 = DecoKiller.collapse_map(ptrace_ancilla(rho2_, ds, da), rho2; pin=cfg.pin)

Cx == I(size(Cx, 1))

If this is true, then the collapse is the identity
true


true

In [160]:
# Step 4: Update noise guess and recovery map
DecoKiller.update_noise_guess!(st, povm)
model = CollisionModel(st.choice.current == 1 ?
                        C1 * Xi1 * st.noise_options[1].supermap :
                        C2 * Xi2 * st.noise_options[2].supermap, 
                        cfg.sigma)
P = kraus_to_superop(model.kraus_rec)

# ======================================
# Step 5: Recovery
rho_rec, _ = apply_collision(model, rho_to_rec; trace=true)
rho_rec

8×8 Matrix{ComplexF64}:
     0.444539+4.2375e-34im   …     0.0914976+0.00104281im
 -3.53671e-18-2.46787e-20im      -1.1524e-17+7.91367e-20im
  2.34199e-17+3.86454e-20im     -1.18264e-17-1.83069e-19im
 -1.50643e-17+7.87107e-20im      4.45068e-18-1.34298e-19im
  6.23557e-19+7.56411e-20im     -1.81896e-17+7.25292e-20im
 -1.08955e-17+3.99453e-20im  …   3.08815e-17-1.25983e-19im
 -1.43608e-17-2.84632e-20im     -5.93021e-18+5.46127e-20im
    0.0914976-0.00104281im          0.446528+3.13069e-34im

In [161]:
evolve(r2, maps1[2]["Xi"])

8×8 Matrix{ComplexF64}:
     0.133295+2.3392e-35im   …     0.0286594+0.0065608im
 -1.37806e-17-4.56027e-19im     -9.45083e-19+8.12155e-19im
 -4.55803e-20+3.90333e-20im     -5.06346e-18-3.78477e-20im
 -5.79742e-18+1.18939e-19im      2.51322e-18+1.43954e-19im
  3.27654e-18-1.03542e-19im     -5.30044e-18-9.3804e-20im
 -6.29564e-18+1.32051e-20im  …   6.37935e-19-9.12625e-20im
   -1.057e-18-9.54217e-19im     -1.00304e-17+4.66662e-19im
    0.0286594-0.0065608im           0.145809-2.41335e-35im

In [162]:
isapprox(rho_to_rec_from_superop, rho_to_rec; atol=1e-12)
isapprox(rho_to_rec_from_superop, evolve(r2, maps1[2]["Xi"]); atol=1e-12)
isapprox(rho_to_rec_from_superop, evolve(st.ρ0, maps1[2]["Nx"]); atol=1e-12)

true

In [163]:
r2_ = unvec(resh(maps1[2]["Nx"])*vec(r1))
isapprox(r2_/tr(r2_), rho_to_rec; atol=1e-9)
r2_

8×8 Matrix{ComplexF64}:
     0.133295-7.70372e-34im  …     0.0286594+0.0065608im
 -1.37806e-17-4.56027e-19im     -9.45083e-19+8.12155e-19im
 -4.55803e-20+3.90333e-20im     -5.06346e-18-3.78477e-20im
 -5.79742e-18+1.18939e-19im      2.51322e-18+1.43954e-19im
  3.27654e-18-1.03542e-19im     -5.30044e-18-9.3804e-20im
 -6.29564e-18+1.32051e-20im  …   6.37935e-19-9.12625e-20im
   -1.057e-18-9.54217e-19im     -1.00304e-17+4.66662e-19im
    0.0286594-0.0065608im           0.145809+0.0im

In [164]:
r2_rec = evolve(r2_, maps[2]["P"])
r3 = unvec(resh(maps1[2]["P"])*resh(maps1[2]["Nx"])*vec(r1))
isapprox(r3/tr(r3), r2_rec; atol=1e-14)

true

In [165]:
println(DecoKiller.fidelity(r3, st.ρ0))
println(DecoKiller.fidelity(r2_rec, st.ρ0))
println(DecoKiller.fidelity(r2_, st.ρ0))

0.4650145861882324
0.46501458693198955
0.15431098506076596


In [166]:
using JSON

In [167]:
results_path = joinpath(folder, "data", "results.json")
results = JSON.parsefile(results_path)

JSON.Object{String, Any} with 8 entries:
  "avg_fidelities"     => Any[0.637228, 0.600849]
  "avg_ref_fidelities" => Any[0.244303, 0.180602]
  "choice_c1"          => Any[1]
  "choice_c2"          => Any[0]
  "choice_history"     => Any[1, 1, 1, 1, 2, 1, 1, 1, 1, 1  …  2, 1, 1, 1, 2, 1…
  "codespace_overlaps" => Any[Any[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, …
  "fidelities"         => Any[0.518371, 0.465015, 0.511076, 0.456677, 0.712468,…
  "ref_fidelities"     => Any[0.225699, 0.147394, 0.224557, 0.145355, 0.256079,…